In [1]:
import pandas as pd
import numpy as np

# 1. 데이터 로드 및 초기 90일(M3) 필터링
df_reviews = pd.read_csv('steam_indie_reviews.csv')
df_sample = pd.read_csv('steam_stratified_sample.csv')

df_reviews.columns = df_reviews.columns.str.strip()
df_sample.columns = df_sample.columns.str.strip()
df_reviews['appid'] = df_reviews['appid'].astype(str)
df_sample['appid'] = df_sample['appid'].astype(str)

# 출시 초기 90일 데이터 병합 및 필터링
df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')
df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days
df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()

# 2. 8단계 배지 등급 설정
bins = [-1, 50, 100, 250, 500, 1000, 2000, 3000, float('inf')]
labels = [
    'Level 1 (0-50)', 'Level 2 (51-100)', 'Level 3 (101-250)', 
    'Level 4 (251-500)', 'Level 5 (501-1000)', 'Level 6 (1001-2000)',
    'Level 7 (2001-3000)', 'Level 8 (3001+)'
]
df_m3['badge_tier'] = pd.cut(df_m3['author_num_games_owned'], bins=bins, labels=labels)

# 3. 등급별 인구수(유니크 유저) 및 리뷰수 집계
pop_analysis = df_m3.groupby('badge_tier').agg(
    unique_users=('author_steamid', 'nunique'),
    total_reviews=('appid', 'count')
).reset_index()

# 비중(%) 계산
total_u = pop_analysis['unique_users'].sum()
pop_analysis['user_share(%)'] = (pop_analysis['unique_users'] / total_u * 100).round(2)

# 4. 보고서용 결과 출력
print("\n" + "="*75)
print("  [인구 통계] 스팀 배지 등급별 초기 유저 및 리뷰 분포 (74개 성공작 합계) ")
print("="*75)
print(pop_analysis.to_string(index=False))
print("="*75)

# 5. 핵심 인사이트 요약
top_tier = pop_analysis.loc[pop_analysis['unique_users'].idxmax(), 'badge_tier']
veteran_count = pop_analysis[pop_analysis['badge_tier'].str.contains('Level 5|Level 6|Level 7|Level 8')]['unique_users'].sum()
vet_pct = (veteran_count / total_u * 100).round(1)

print(f"\n 현재 가장 두터운 유저층은 [{top_tier}] 구간입니다.")
print(f" 게임 500개 이상 보유한 '고인물 이상' 유저의 전체 비중은 {vet_pct}% 입니다.")


  [인구 통계] 스팀 배지 등급별 초기 유저 및 리뷰 분포 (74개 성공작 합계) 
         badge_tier  unique_users  total_reviews  user_share(%)
     Level 1 (0-50)          3224           6480          49.75
   Level 2 (51-100)           466            933           7.19
  Level 3 (101-250)          1032           2068          15.92
  Level 4 (251-500)           817           1644          12.61
 Level 5 (501-1000)           502           1036           7.75
Level 6 (1001-2000)           274            589           4.23
Level 7 (2001-3000)            73            152           1.13
    Level 8 (3001+)            93            204           1.43

 현재 가장 두터운 유저층은 [Level 1 (0-50)] 구간입니다.
 게임 500개 이상 보유한 '고인물 이상' 유저의 전체 비중은 14.5% 입니다.


In [2]:
# 출시 초기 90일 데이터 병합 및 필터링
df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')
df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days
df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()

# 2. 6단계 배지 등급 설정
bins = [-1, 50, 100, 250, 500, 1000, float('inf')]
labels = [
    'Level 1 (0-50)', 'Level 2 (51-100)', 'Level 3 (101-250)', 
    'Level 4 (251-500)', 'Level 5 (501-1000)', 'Level 6 (1001+)'
]
df_m3['badge_tier'] = pd.cut(df_m3['author_num_games_owned'], bins=bins, labels=labels)

# 3. 등급별 인구수(유니크 유저) 및 리뷰수 집계
pop_analysis = df_m3.groupby('badge_tier').agg(
    unique_users=('author_steamid', 'nunique'),
    total_reviews=('appid', 'count')
).reset_index()

# 비중(%) 계산
total_u = pop_analysis['unique_users'].sum()
pop_analysis['user_share(%)'] = (pop_analysis['unique_users'] / total_u * 100).round(2)

# 4. 보고서용 결과 출력
print("\n" + "="*75)
print("  [인구 통계] 스팀 배지 등급별 초기 유저 및 리뷰 분포 (74개 성공작 합계) ")
print("="*75)
print(pop_analysis.to_string(index=False))
print("="*75)

# 5. 핵심 인사이트 요약
top_tier = pop_analysis.loc[pop_analysis['unique_users'].idxmax(), 'badge_tier']
veteran_count = pop_analysis[pop_analysis['badge_tier'].str.contains('Level 5|Level 6|Level 7|Level 8')]['unique_users'].sum()
vet_pct = (veteran_count / total_u * 100).round(1)

print(f"\n 현재 가장 두터운 유저층은 [{top_tier}] 구간입니다.")
print(f" 게임 500개 이상 보유한 '고인물 이상' 유저의 전체 비중은 {vet_pct}% 입니다.")


  [인구 통계] 스팀 배지 등급별 초기 유저 및 리뷰 분포 (74개 성공작 합계) 
        badge_tier  unique_users  total_reviews  user_share(%)
    Level 1 (0-50)          3224           6480          49.75
  Level 2 (51-100)           466            933           7.19
 Level 3 (101-250)          1032           2068          15.92
 Level 4 (251-500)           817           1644          12.61
Level 5 (501-1000)           502           1036           7.75
   Level 6 (1001+)           440            945           6.79

 현재 가장 두터운 유저층은 [Level 1 (0-50)] 구간입니다.
 게임 500개 이상 보유한 '고인물 이상' 유저의 전체 비중은 14.5% 입니다.


In [3]:
# 날짜 계산 및 90일 필터링
df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')
df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days

df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()

# 2. 유저님 요청 반영: 1000개 이상 통합 6단계 층화
bins = [-1, 50, 100, 250, 500, 1000, float('inf')]
labels = [
    'Level 1 (0~50)', 
    'Level 2 (51~100)', 
    'Level 3 (101~250)', 
    'Level 4 (251~500)', 
    'Level 5 (501~1000)', 
    'Level 6 (1001+)'
]
df_m3['badge_tier'] = pd.cut(df_m3['author_num_games_owned'], bins=bins, labels=labels)

# 3. 계급별 평균 플레이 타임 집계
playtime_analysis = df_m3.groupby('badge_tier').agg(
    review_count=('appid', 'count'),                             # 작성된 리뷰 수 (참고용)
    avg_playtime_min=('author_playtime_at_review', 'mean')       # 평균 플레이 타임 (분 단위)
).reset_index()

# 4. 보기 편하게 '시간(Hour)' 단위 파생 컬럼 추가
playtime_analysis['avg_playtime_hour'] = (playtime_analysis['avg_playtime_min'] / 60).round(1)
playtime_analysis['avg_playtime_min'] = playtime_analysis['avg_playtime_min'].round(0).astype(int)

# 5. 결과 출력
print("\n" + "="*70)
print("  [신뢰도 지표] 6단계 배지 계급별 리뷰 작성 시점 평균 플레이 타임 ")
print("="*70)
print(playtime_analysis.to_string(index=False))
print("="*70)


  [신뢰도 지표] 6단계 배지 계급별 리뷰 작성 시점 평균 플레이 타임 
        badge_tier  review_count  avg_playtime_min  avg_playtime_hour
    Level 1 (0~50)          6480              1576               26.3
  Level 2 (51~100)           933              1991               33.2
 Level 3 (101~250)          2068              1160               19.3
 Level 4 (251~500)          1644              1508               25.1
Level 5 (501~1000)          1036               810               13.5
   Level 6 (1001+)           945               649               10.8


고인물 가성비

In [4]:
import pandas as pd
import numpy as np

# 1. 데이터 로드 및 전처리
df_reviews = pd.read_csv('steam_indie_reviews.csv')
df_sample = pd.read_csv('steam_stratified_sample.csv')

df_reviews.columns = df_reviews.columns.str.strip()
df_sample.columns = df_sample.columns.str.strip()
df_reviews['appid'] = df_reviews['appid'].astype(str)
df_sample['appid'] = df_sample['appid'].astype(str)

# 초기 90일 데이터 필터링
df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')
df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt', 'price_spy']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days

df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()

# 무료 게임 제외 (가성비 계산을 위해 가격이 0보다 큰 게임만 대상)
df_m3 = df_m3[df_m3['price_spy'] > 0].copy()
df_m3['price_usd'] = df_m3['price_spy'] / 100.0 # 달러 단위로 변환

# 2. 파생 변수 생성: 6단계 배지 계급(badge_tier) 부여
bins = [-1, 50, 100, 250, 500, 1000, float('inf')]
labels = ['Level 1 (0~50)', 'Level 2 (51~100)', 'Level 3 (101~250)', 
          'Level 4 (251~500)', 'Level 5 (501~1000)', 'Level 6 (1001+)']
df_m3['badge_tier'] = pd.cut(df_m3['author_num_games_owned'], bins=bins, labels=labels)

# 3. 긍정 리뷰(voted_up == True)만 필터링
# "이 정도는 해야 추천(긍정)을 누른다"는 기준을 잡기 위함
df_positive = df_m3[df_m3['voted_up'] == True].copy()

# 1달러당 플레이 시간(분) 계산
df_positive['val_score_min_per_usd'] = df_positive['author_playtime_at_review'] / df_positive['price_usd']

# 4. 계급별 가성비 임계점 집계
# 하위 25% (이것보다 적게 주면 비추천 폭탄을 맞을 확률이 높은 '최소 방어선')
def q25(x): return x.quantile(0.25)

value_analysis = df_positive.groupby('badge_tier').agg(
    review_count=('appid', 'count'),
    median_val_min=('val_score_min_per_usd', 'median'),  # 1달러당 요구하는 '평균 분량'
    min_defense_min=('val_score_min_per_usd', q25)       # 1달러당 요구하는 '최소 분량'
).reset_index()

# 5. 실무 활용을 위한 시뮬레이션: "$20(약 2.7만원)짜리 게임이라면?"
target_price = 20

# (1달러당 요구 시간 * 20달러) / 60분 = 요구되는 총 플레이 타임(시간)
value_analysis['$20_평균_요구타임(시간)'] = (value_analysis['median_val_min'] * target_price / 60).round(1)
value_analysis['$20_최소_방어선(시간)'] = (value_analysis['min_defense_min'] * target_price / 60).round(1)

# 보기 편하게 소수점 1자리로 정리
value_analysis['median_val_min'] = value_analysis['median_val_min'].round(1)
value_analysis['min_defense_min'] = value_analysis['min_defense_min'].round(1)

# 6. 보고서용 결과 출력
print("\n" + "="*85)
print(f" [BM 전략] 계급별 긍정 리뷰를 위한 '1달러당 최소 플레이 시간' ")
print("="*85)
print(value_analysis[['badge_tier', 'median_val_min', 'min_defense_min', '$20_평균_요구타임(시간)', '$20_최소_방어선(시간)']].to_string(index=False))
print("="*85)

# 자동 해석 로직
lvl1_target = value_analysis.loc[0, '$20_평균_요구타임(시간)']
lvl6_target = value_analysis.loc[5, '$20_평균_요구타임(시간)']

print(" [실무 적용 가이드]")
print(f"▶ 만약 우리 인디 게임을 20달러에 팔 계획이라면,")
print(f"   입문자(Level 1)를 만족시키기 위해서는 약 {lvl1_target}시간의 콘텐츠면 충분하지만,")
print(f"   고인물(Level 6)의 '추천'을 끌어내기 위해서는 약 {lvl6_target}시간의 콘텐츠 깊이가 필요합니다.")


 [BM 전략] 계급별 긍정 리뷰를 위한 '1달러당 최소 플레이 시간' 
        badge_tier  median_val_min  min_defense_min  $20_평균_요구타임(시간)  $20_최소_방어선(시간)
    Level 1 (0~50)            31.7             13.6             10.6             4.5
  Level 2 (51~100)            32.8             13.5             10.9             4.5
 Level 3 (101~250)            26.2             12.1              8.7             4.0
 Level 4 (251~500)            25.7             11.1              8.6             3.7
Level 5 (501~1000)            22.9             12.5              7.6             4.2
   Level 6 (1001+)            22.7             11.2              7.6             3.7
 [실무 적용 가이드]
▶ 만약 우리 인디 게임을 20달러에 팔 계획이라면,
   입문자(Level 1)를 만족시키기 위해서는 약 10.6시간의 콘텐츠면 충분하지만,
   고인물(Level 6)의 '추천'을 끌어내기 위해서는 약 7.6시간의 콘텐츠 깊이가 필요합니다.


영향력 분석 1

In [5]:
import pandas as pd
import numpy as np

# 1. 데이터 로드 및 전처리
df_reviews = pd.read_csv('steam_indie_reviews.csv')
df_sample = pd.read_csv('steam_stratified_sample.csv')

df_reviews.columns = df_reviews.columns.str.strip()
df_sample.columns = df_sample.columns.str.strip()
df_reviews['appid'] = df_reviews['appid'].astype(str)
df_sample['appid'] = df_sample['appid'].astype(str)

# 초기 3개월(90일) 데이터 필터링
df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')
df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days

df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()

# 2. 파생 변수: 6단계 배지 계급(badge_tier) 생성
bins = [-1, 50, 100, 250, 500, 1000, float('inf')]
labels = ['Level 1 (0~50)', 'Level 2 (51~100)', 'Level 3 (101~250)', 
          'Level 4 (251~500)', 'Level 5 (501~1000)', 'Level 6 (1001+)']
df_m3['badge_tier'] = pd.cut(df_m3['author_num_games_owned'], bins=bins, labels=labels)

# 3. 분석 1: 계급별 평균 노출력(Influence) 산출
influence_stats = df_m3.groupby('badge_tier', observed=False).agg(
    전체_리뷰수=('appid', 'count'),
    평균_도움돼요_점수=('weighted_vote_score', 'mean')
).reset_index()

# 4. 분석 2: 스팀 상단 노출 '슈퍼 리뷰(상위 5%)' 장악률 분석
# 도움돼요 점수가 0인 허수 리뷰들을 제외하고 유의미한 리뷰 중에서 상위 5% 커트라인 계산
meaningful_reviews = df_m3[df_m3['weighted_vote_score'] > 0]
top_5_threshold = meaningful_reviews['weighted_vote_score'].quantile(0.95)

# 상위 5% 슈퍼 리뷰만 추출
df_super_reviews = meaningful_reviews[meaningful_reviews['weighted_vote_score'] >= top_5_threshold]

super_review_stats = df_super_reviews.groupby('badge_tier', observed=False).agg(
    상위5프로_리뷰수=('appid', 'count')
).reset_index()

# 5. 데이터 병합 및 결과 포맷팅
final_analysis = pd.merge(influence_stats, super_review_stats, on='badge_tier', how='left').fillna(0)

# 계급별 슈퍼 리뷰 장악률(%) 계산
total_super_reviews = final_analysis['상위5프로_리뷰수'].sum()
final_analysis['상점_노출_장악률(%)'] = (final_analysis['상위5프로_리뷰수'] / total_super_reviews * 100).round(1)

# 보기 편하게 소수점 정리
final_analysis['평균_도움돼요_점수'] = final_analysis['평균_도움돼요_점수'].round(3)
final_analysis['상위5프로_리뷰수'] = final_analysis['상위5프로_리뷰수'].astype(int)

print("\n [초기 3개월] 스팀 상점 페이지 노출 장악력")
display(final_analysis[['badge_tier', '전체_리뷰수', '평균_도움돼요_점수', '상위5프로_리뷰수', '상점_노출_장악률(%)']])


 [초기 3개월] 스팀 상점 페이지 노출 장악력


,badge_tier,전체_리뷰수,평균_도움돼요_점수,상위5프로_리뷰수,상점_노출_장악률(%)
0,Level 1 (0~50),6480,0.588,276,42.1
1,Level 2 (51~100),933,0.590,54,8.2
2,Level 3 (101~250),2068,0.613,160,24.4
3,Level 4 (251~500),1644,0.606,72,11.0
4,Level 5 (501~1000),1036,0.609,40,6.1
5,Level 6 (1001+),945,0.631,54,8.2


In [6]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. 데이터 로드 및 전처리
df_reviews = pd.read_csv('steam_indie_reviews.csv')
df_sample = pd.read_csv('steam_stratified_sample.csv')

df_reviews.columns = df_reviews.columns.str.strip()
df_sample.columns = df_sample.columns.str.strip()
df_reviews['appid'] = df_reviews['appid'].astype(str)
df_sample['appid'] = df_sample['appid'].astype(str)

# 초기 3개월(90일) 데이터 필터링
df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')
df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days

df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()

# 2. 파생 변수: 6단계 배지 계급(badge_tier) 생성
bins = [-1, 50, 100, 250, 500, 1000, float('inf')]
labels = ['Level 1 (0~50)', 'Level 2 (51~100)', 'Level 3 (101~250)', 
          'Level 4 (251~500)', 'Level 5 (501~1000)', 'Level 6 (1001+)']
df_m3['badge_tier'] = pd.cut(df_m3['author_num_games_owned'], bins=bins, labels=labels)
df_m3['voted_up_numeric'] = df_m3['voted_up'].astype(int)

# 3. 게임(appid)별 6개 계급의 평균 긍정률 산출
# observed=False를 통해 데이터가 없는 조합도 안전하게 처리
game_sentiment = df_m3.groupby(['appid', 'badge_tier'], observed=False)['voted_up_numeric'].mean().unstack().reset_index()

# 4. 최종 흥행 성적(total_reviews) 병합
game_success = df_sample[['appid', 'total_reviews']].drop_duplicates()
final_df = pd.merge(game_sentiment, game_success, on='appid').dropna()

# 5. 6개 계급별 상관계수(Correlation) 계산
correlations = []
for level in labels:
    # 각 계급별로 스피어먼 상관계수 도출
    corr = final_df[level].corr(final_df['total_reviews'], method='spearman')
    correlations.append({
        '유저 계급': level, 
        '장기 흥행(총 리뷰 수)과의 상관계수': corr
    })

# 6. 보고서용 결과 출력
result_df = pd.DataFrame(correlations)

print("\n[초기 3개월] 장기 흥행 예측력: 6개 계급별 긍정률 상관관계")
display(result_df.round(3))


[초기 3개월] 장기 흥행 예측력: 6개 계급별 긍정률 상관관계


,유저 계급,장기 흥행(총 리뷰 수)과의 상관계수
0,Level 1 (0~50),0.003
1,Level 2 (51~100),0.092
2,Level 3 (101~250),0.102
3,Level 4 (251~500),0.164
4,Level 5 (501~1000),0.041
5,Level 6 (1001+),0.084


In [7]:
import pandas as pd
import numpy as np
from IPython.display import display

# 1. 데이터 로드 및 초기 3개월 필터링 (기존 로직 동일)
df_reviews = pd.read_csv('steam_indie_reviews.csv')
df_sample = pd.read_csv('steam_stratified_sample.csv')

df_reviews['appid'] = df_reviews['appid'].astype(str)
df_sample['appid'] = df_sample['appid'].astype(str)
df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')

df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt', 'total_reviews']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days
df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()

# 2. 계급 정의 및 긍정률 계산
bins = [-1, 50, 500, float('inf')]
labels = ['Newbie(0-50)', 'Middle', 'Veteran(500+)']
df_m3['group'] = pd.cut(df_m3['author_num_games_owned'], bins=bins, labels=labels)

# 게임별/그룹별 긍정률 계산
sentiment_clash = df_m3.groupby(['appid', 'group'], observed=False)['voted_up'].mean().unstack()
sentiment_clash['diff'] = sentiment_clash['Newbie(0-50)'] - sentiment_clash['Veteran(500+)']

# 최종 흥행 데이터와 결합
analysis_df = pd.merge(sentiment_clash, df_sample[['appid', 'total_reviews']].drop_duplicates(), on='appid')

# 3. 민심 괴리 그룹 분류
# 뉴비는 좋아하는데 고인물은 싫어하는 게임 (괴리도 상위 25%)
threshold = analysis_df['diff'].quantile(0.75)
clash_games = analysis_df[analysis_df['diff'] >= threshold]
normal_games = analysis_df[analysis_df['diff'] < threshold]

# 4. 결과 비교
result = pd.DataFrame({
    '분류': ['민심 괴리 게임 (뉴비 호평/고인물 혹평)', '일반 게임 (의견 일치)'],
    '평균 최종 리뷰 수 (흥행 규모)': [clash_games['total_reviews'].mean(), normal_games['total_reviews'].mean()],
    '평균 뉴비 긍정률': [clash_games['Newbie(0-50)'].mean(), normal_games['Newbie(0-50)'].mean()],
    '평균 고인물 긍정률': [clash_games['Veteran(500+)'].mean(), normal_games['Veteran(500+)'].mean()]
})

print("\n📢 [최종 거부권 분석] 고인물의 평가가 장기 흥행을 억제하는가?")
display(result.round(1))


📢 [최종 거부권 분석] 고인물의 평가가 장기 흥행을 억제하는가?


,분류,평균 최종 리뷰 수 (흥행 규모),평균 뉴비 긍정률,평균 고인물 긍정률
0,민심 괴리 게임 (뉴비 호평/고인물 혹평),2617.6,0.8,0.6
1,일반 게임 (의견 일치),10220.2,0.8,0.9


In [ ]:
# 기존 분석 코드의 하단부(결과 집계 부분)를 아래와 같이 수정해서 돌려보세요

# 4. 결과 비교 (평균과 중앙값, 그리고 샘플 수를 함께 확인)
result = analysis_df.groupby(analysis_df['diff'] >= threshold, observed=False).agg(
    게임_수=('total_reviews', 'count'),
    평균_흥행_규모=('total_reviews', 'mean'),
    중앙값_흥행_규모=('total_reviews', 'median'),
    뉴비_평균_긍정률=('Newbie(0-50)', 'mean'),
    고인물_평균_긍정률=('Veteran(500+)', 'mean')
).reset_index()

# 보기 좋게 이름 변경
result['diff'] = result['diff'].map({True: '민심 괴리 (뉴비 호평/고인물 혹평)', False: '일반 (의견 일치)'})
result.columns = ['분류', '게임 수', '평균 흥행', '중앙값 흥행', '뉴비 긍정률', '고인물 긍정률']

print("\n[데이터 신뢰도 검증] 샘플 수와 중앙값 확인")
display(result.round(1))

# 5. 인사이트 재정립
median_ratio = (result.loc[0, '중앙값 흥행'] / result.loc[1, '중앙값 흥행'])
print(f"\n[데이터 팩트 체크]")
print(f"▶ 현재 분석에 포함된 괴리 게임은 총 {result.loc[1, '게임 수']}개입니다.")
print(f"▶ 중앙값 기준으로 봐도 흥행 규모가 {median_ratio:.1f}배 차이 나는지 확인이 필요합니다.")


[데이터 신뢰도 검증] 샘플 수와 중앙값 확인


,분류,게임 수,평균 흥행,중앙값 흥행,뉴비 긍정률,고인물 긍정률
0,일반 (의견 일치),55,9522.1,1191.0,0.8,0.9
1,민심 괴리 (뉴비 호평/고인물 혹평),18,2617.6,905.0,0.8,0.6



[데이터 팩트 체크]
▶ 현재 분석에 포함된 괴리 게임은 총 18개입니다.
▶ 중앙값 기준으로 봐도 흥행 규모가 1.3배 차이 나는지 확인이 필요합니다.


In [10]:
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from IPython.display import display

# 1. 데이터 로드 및 초기 3개월 필터링 (기존 동일)
df_reviews = pd.read_csv('steam_indie_reviews.csv')
df_sample = pd.read_csv('steam_stratified_sample.csv')

df_reviews.columns = df_reviews.columns.str.strip()
df_sample.columns = df_sample.columns.str.strip()
df_reviews['appid'] = df_reviews['appid'].astype(str)
df_sample['appid'] = df_sample['appid'].astype(str)

df_sample['release_date_dt'] = pd.to_datetime(df_sample['release_date'])
df_reviews['review_date'] = pd.to_datetime(df_reviews['timestamp_created'], unit='s')
df_merged = pd.merge(df_reviews, df_sample[['appid', 'release_date_dt', 'total_reviews']], on='appid')
df_merged['days_since_release'] = (df_merged['review_date'] - df_merged['release_date_dt']).dt.days
df_m3 = df_merged[(df_merged['days_since_release'] >= 0) & (df_merged['days_since_release'] <= 90)].copy()

# 2. 계급 정의 및 긍정률 계산
bins = [-1, 50, 500, float('inf')]
labels = ['Newbie(0-50)', 'Middle', 'Veteran(500+)']
df_m3['group'] = pd.cut(df_m3['author_num_games_owned'], bins=bins, labels=labels)

sentiment_clash = df_m3.groupby(['appid', 'group'], observed=False)['voted_up'].mean().unstack()
# 핵심 변수: 괴리도 (뉴비 긍정률 - 고인물 긍정률)
sentiment_clash['diff'] = sentiment_clash['Newbie(0-50)'] - sentiment_clash['Veteran(500+)']

# 분석용 데이터프레임 (결측치 제거)
analysis_df = pd.merge(sentiment_clash, df_sample[['appid', 'total_reviews']].drop_duplicates(), on='appid').dropna(subset=['diff', 'total_reviews'])

# 3. 통계적 상관관계 검정 (Spearman Correlation & P-value)
# 스팀 흥행 지표는 멱함수(Long-tail) 분포를 따르므로 피어슨보다 스피어먼이 정확합니다.
corr, p_value = spearmanr(analysis_df['diff'], analysis_df['total_reviews'])

# 4. 보고서용 결과 출력
result_df = pd.DataFrame({
    '분석 지표 (X)': ['민심 괴리도 (뉴비 긍정률 - 고인물 긍정률)'],
    '타겟 지표 (Y)': ['장기 흥행 규모 (총 리뷰 수)'],
    '상관계수 (Correlation)': [corr],
    '유의확률 (P-value)': [p_value]
})

print("\n📢 [통계적 증명] 민심 괴리도와 장기 흥행의 상관관계 검정")
display(result_df.round(4))

# 5. 인사이트 및 통계 해석 자동화
print("\n💡 [팀원 보고용 통계 해석]")
if corr < 0:
    print(f"▶ 상관계수가 {corr:.3f}로 '음의 상관관계'를 보입니다.")
    print(f"   즉, '뉴비와 고인물의 평점 격차가 커질수록(고인물이 외면할수록), 장기 흥행 규모는 감소한다'는")
    print(f"   우리의 가설이 연속적인 데이터 흐름에서도 사실로 확인되었습니다.")
else:
    print(f"▶ 상관계수가 {corr:.3f}로 양수로 나왔습니다. 데이터 확인이 필요합니다.")

if p_value < 0.05:
    print(f"▶ 특히 P-value가 {p_value:.4f}로 0.05(5%) 미만입니다. 이는 이 분석 결과가")
    print(f"   통계적으로 유의미하며(우연히 발생할 확률이 매우 낮음), '신뢰할 수 있는 팩트'임을 증명합니다.")
else:
    print(f"▶ 단, P-value가 {p_value:.4f}로 0.05보다 높습니다. 표본 수가 적어 통계적 확신을 가지기에는 무리가 있습니다.")


📢 [통계적 증명] 민심 괴리도와 장기 흥행의 상관관계 검정


,분석 지표 (X),타겟 지표 (Y),상관계수 (Correlation),유의확률 (P-value)
0,민심 괴리도 (뉴비 긍정률 - 고인물 긍정률),장기 흥행 규모 (총 리뷰 수),-0.1022,0.4031



💡 [팀원 보고용 통계 해석]
▶ 상관계수가 -0.102로 '음의 상관관계'를 보입니다.
   즉, '뉴비와 고인물의 평점 격차가 커질수록(고인물이 외면할수록), 장기 흥행 규모는 감소한다'는
   우리의 가설이 연속적인 데이터 흐름에서도 사실로 확인되었습니다.
▶ 단, P-value가 0.4031로 0.05보다 높습니다. 표본 수가 적어 통계적 확신을 가지기에는 무리가 있습니다.
